# Polynomial Regression vs GAM

**Topics:** Non-linear Regression, Polynomials, Smooth Terms, Model Selection

## Overview

This notebook compares two approaches for modeling non-linear relationships: polynomial regression and Generalized Additive Models (GAM). We'll explore their strengths, weaknesses, and when to use each.

## What You'll Learn

- Recognize non-linear relationships in data
- Fit polynomial regression (quadratic, cubic)
- Fit GAM with smooth terms
- Compare model flexibility and overfitting
- Select optimal polynomial degree
- Understand advantages of GAM over polynomials
- Make informed model choices

---

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from aurora.models import fit_glm
from aurora.models.gam import fit_gam
from aurora.smoothing.splines.cubic import CubicSplineBasis
from aurora.validation.metrics import root_mean_squared_error, aic, bic, r_squared

sns.set_style('whitegrid')
np.random.seed(42)

# Aliases for convenience
rmse = root_mean_squared_error

## 2. Generate Non-linear Data

Simulate engine efficiency vs RPM with realistic non-linear pattern:

In [ ]:
n = 200

# RPM range: 1000-6000
rpm = np.random.uniform(1000, 6000, n)

# True non-linear relationship (inverted U-shape with bump)
# Peak efficiency around 3500 RPM
rpm_scaled = (rpm - 3500) / 1000
true_efficiency = (
    75  # Base efficiency
    - 3 * rpm_scaled**2  # Quadratic penalty away from optimum
    + 2 * np.sin(rpm_scaled * 2)  # Local oscillation
    - 0.5 * rpm_scaled**3  # Asymmetry at extremes
)

# Add noise
noise = np.random.randn(n) * 2
efficiency = true_efficiency + noise

# Create DataFrame
df = pd.DataFrame({
    'rpm': rpm,
    'efficiency': efficiency,
    'true_efficiency': true_efficiency
})

print(f"Generated data for {n} engine measurements")
print(f"\nRPM range: {df['rpm'].min():.0f} - {df['rpm'].max():.0f}")
print(f"Efficiency range: {df['efficiency'].min():.1f}% - {df['efficiency'].max():.1f}%")

# Plot data
plt.figure(figsize=(10, 6))
plt.scatter(df['rpm'], df['efficiency'], alpha=0.5, s=30, label='Observed', edgecolor='k', linewidth=0.5)
sorted_idx = np.argsort(df['rpm'])
plt.plot(df['rpm'].values[sorted_idx], df['true_efficiency'].values[sorted_idx], 
         'r-', lw=3, label='True relationship')
plt.xlabel('Engine RPM')
plt.ylabel('Fuel Efficiency (%)')
plt.title('Engine Efficiency vs RPM\n(Complex non-linear relationship)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Baseline: Linear Model

First, try a simple linear model (will fail!):

In [ ]:
# Standardize RPM for numerical stability
rpm_std = (df['rpm'] - df['rpm'].mean()) / df['rpm'].std()
y = df['efficiency'].values

# Fit linear model
X_linear = rpm_std.values.reshape(-1, 1)  # Convert to numpy and reshape
result_linear = fit_glm(X=X_linear, y=y, family='gaussian')

# Predictions
y_pred_linear = result_linear.predict(X_linear)

# Plot
plt.figure(figsize=(10, 6))
plt.scatter(df['rpm'], df['efficiency'], alpha=0.4, s=30, label='Data', color='gray')
plt.plot(df['rpm'].values[sorted_idx], df['true_efficiency'].values[sorted_idx], 
         'g-', lw=3, label='True relationship', zorder=5)
plt.plot(df['rpm'].values[sorted_idx], y_pred_linear[sorted_idx], 
         'b--', lw=2, label='Linear fit', alpha=0.7)
plt.xlabel('Engine RPM')
plt.ylabel('Fuel Efficiency (%)')
plt.title('Linear Model: Cannot Capture Non-linearity')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate R² using aurora.validation.metrics
r2_linear = r_squared(y, y_pred_linear)

print(f"\nLinear Model:")
print(f"  R² = {r2_linear:.4f}")
print(f"  RMSE = {rmse(y, y_pred_linear):.2f}")
print(f"  AIC = {result_linear.aic_:.1f}")

## 4. Polynomial Regression

### Fit Multiple Polynomial Degrees

In [ ]:
# Try polynomial degrees 2-10
degrees = [2, 3, 4, 5, 6, 8, 10]
poly_results = {}

# Convert to numpy array
rpm_std_np = rpm_std.values

for deg in degrees:
    # Create polynomial features (let fit_glm add intercept)
    X_poly = np.column_stack([rpm_std_np**p for p in range(1, deg+1)])
    
    # Fit model
    result = fit_glm(X=X_poly, y=y, family='gaussian')
    y_pred = result.predict(X_poly)
    
    poly_results[deg] = {
        'result': result,
        'y_pred': y_pred,
        'rmse': rmse(y, y_pred),
        'r2': r_squared(y, y_pred),
        'aic': result.aic_,
        'bic': result.bic_,
        'n_params': deg + 1
    }

# Summary table
poly_summary = pd.DataFrame({
    'Degree': degrees,
    'Params': [poly_results[d]['n_params'] for d in degrees],
    'R²': [poly_results[d]['r2'] for d in degrees],
    'RMSE': [poly_results[d]['rmse'] for d in degrees],
    'AIC': [poly_results[d]['aic'] for d in degrees],
    'BIC': [poly_results[d]['bic'] for d in degrees]
})

print("\nPolynomial Regression Results:")
print(poly_summary.to_string(index=False))
print(f"\nBest by AIC: Degree {poly_summary.loc[poly_summary['AIC'].idxmin(), 'Degree']:.0f}")
print(f"Best by BIC: Degree {poly_summary.loc[poly_summary['BIC'].idxmin(), 'Degree']:.0f}")

### Visualize Different Polynomial Degrees

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

degrees_to_plot = [2, 3, 4, 5, 8, 10]

for idx, deg in enumerate(degrees_to_plot):
    ax = axes[idx]
    
    # Data
    ax.scatter(df['rpm'], df['efficiency'], alpha=0.3, s=20, color='gray')
    
    # True relationship
    ax.plot(df['rpm'].values[sorted_idx], df['true_efficiency'].values[sorted_idx], 
            'g-', lw=2, label='True', alpha=0.7)
    
    # Polynomial fit
    ax.plot(df['rpm'].values[sorted_idx], poly_results[deg]['y_pred'][sorted_idx],
            'r-', lw=2, label=f'Degree {deg}')
    
    ax.set_xlabel('RPM')
    ax.set_ylabel('Efficiency (%)')
    ax.set_title(f'Polynomial Degree {deg}\nR² = {poly_results[deg]["r2"]:.4f}, AIC = {poly_results[deg]["aic"]:.1f}')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nObservations:")
print("  • Degree 2-3: Underfit (too smooth)")
print("  • Degree 4-5: Good balance")
print("  • Degree 8-10: Overfit (wiggly at edges)")

## 5. Problems with High-Degree Polynomials

### Edge Effects (Runge's Phenomenon)

In [ ]:
# Extend range to see edge effects
rpm_extended = np.linspace(500, 6500, 300)
rpm_ext_std = (rpm_extended - df['rpm'].mean()) / df['rpm'].std()

# Predict with different polynomial degrees
fig, ax = plt.subplots(figsize=(12, 6))

for deg in [3, 5, 10]:
    # Match the structure used in training (without manual intercept)
    X_ext = np.column_stack([rpm_ext_std**p for p in range(1, deg+1)])
    y_ext = poly_results[deg]['result'].predict(X_ext)
    ax.plot(rpm_extended, y_ext, lw=2, label=f'Degree {deg}', alpha=0.7)

# Original data range
ax.axvline(df['rpm'].min(), color='gray', linestyle='--', alpha=0.5, label='Data range')
ax.axvline(df['rpm'].max(), color='gray', linestyle='--', alpha=0.5)
ax.scatter(df['rpm'], df['efficiency'], alpha=0.3, s=20, color='black', zorder=5)

ax.set_xlabel('Engine RPM')
ax.set_ylabel('Fuel Efficiency (%)')
ax.set_title('Polynomial Edge Effects: Extrapolation Danger!\n(High-degree polynomials behave badly outside data range)')
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(50, 90)
plt.tight_layout()
plt.show()

## 6. GAM with Smooth Terms

### Automatic Smoothness Selection via GCV

In [ ]:
# Fit GAM with smooth term
# fit_gam automatically creates the basis and selects smoothing parameter
result_gam = fit_gam(
    x=rpm_std.values,
    y=y,
    n_basis=15,
    basis_type='cubic'
)

y_pred_gam = result_gam.fitted_values

# Calculate R² using aurora.validation.metrics
r2_gam = r_squared(y, y_pred_gam)

print("GAM Results:")
print(f"  Optimal lambda: {result_gam.lambda_:.4f}")
print(f"  Effective degrees of freedom: {result_gam.edf:.2f}")
print(f"  R²: {r2_gam:.4f}")
print(f"  RMSE: {rmse(y, y_pred_gam):.2f}")
print(f"  GCV score: {result_gam.gcv_score:.4f}")
print(f"\ EDF ≈ {result_gam.edf:.1f} suggests GAM uses ~{result_gam.edf:.0f} effective parameters")
print(f"  (comparable to polynomial degree {result_gam.edf-1:.0f})")

## 7. Compare All Methods

In [ ]:
# Best polynomial (by AIC)
best_poly_deg = int(poly_summary.loc[poly_summary['AIC'].idxmin(), 'Degree'])
y_pred_poly_best = poly_results[best_poly_deg]['y_pred']

# Comparison plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Fits
axes[0].scatter(df['rpm'], df['efficiency'], alpha=0.3, s=20, color='gray', label='Data')
axes[0].plot(df['rpm'].values[sorted_idx], df['true_efficiency'].values[sorted_idx],
             'k-', lw=3, label='True', alpha=0.7, zorder=5)
axes[0].plot(df['rpm'].values[sorted_idx], y_pred_linear[sorted_idx],
             'b--', lw=2, label='Linear', alpha=0.7)
axes[0].plot(df['rpm'].values[sorted_idx], y_pred_poly_best[sorted_idx],
             'r-', lw=2, label=f'Polynomial (deg {best_poly_deg})', alpha=0.7)
axes[0].plot(df['rpm'].values[sorted_idx], y_pred_gam[sorted_idx],
             'g-', lw=2, label=f'GAM (edf={result_gam.edf:.1f})', alpha=0.7)
axes[0].set_xlabel('Engine RPM')
axes[0].set_ylabel('Fuel Efficiency (%)')
axes[0].set_title('Model Comparison: All Fits')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: Residuals
resid_linear = y - y_pred_linear
resid_poly = y - y_pred_poly_best
resid_gam = y - y_pred_gam

axes[1].scatter(y_pred_linear, resid_linear, alpha=0.4, s=30, label='Linear')
axes[1].scatter(y_pred_poly_best, resid_poly, alpha=0.4, s=30, label=f'Polynomial (deg {best_poly_deg})')
axes[1].scatter(y_pred_gam, resid_gam, alpha=0.4, s=30, label='GAM')
axes[1].axhline(0, color='k', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Fitted Values')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Comparison')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Quantitative comparison
comparison_df = pd.DataFrame({
    'Model': ['Linear', f'Polynomial (deg {best_poly_deg})', 'GAM'],
    'Parameters': [2, best_poly_deg+1, f'{result_gam.edf:.1f}'],
    'R²': [
        f'{r2_linear:.4f}',
        f'{poly_results[best_poly_deg]["r2"]:.4f}',
        f'{r2_gam:.4f}'
    ],
    'RMSE': [
        f'{rmse(y, y_pred_linear):.2f}',
        f'{poly_results[best_poly_deg]["rmse"]:.2f}',
        f'{rmse(y, y_pred_gam):.2f}'
    ],
    'Selection Metric': [
        f'AIC: {result_linear.aic_:.1f}',
        f'AIC: {poly_results[best_poly_deg]["aic"]:.1f}',
        f'GCV: {result_gam.gcv_score:.4f}'
    ]
})

print("\nQuantitative Comparison:")
print(comparison_df.to_string(index=False))
print(f"\n GAM achieves best fit with automatic smoothness selection")
print(f" GAM avoids edge effects and overfitting issues of high-degree polynomials")

## 8. Advantages of GAM over Polynomials

### 1. Local Flexibility

In [ ]:
# Create data with local feature
np.random.seed(123)
n_local = 150
x_local = np.random.uniform(0, 10, n_local)

# True function: mostly linear with local bump
y_true_local = 2 * x_local + 3 * np.exp(-((x_local - 5)**2) / 0.5)  # Gaussian bump at x=5
y_local = y_true_local + np.random.randn(n_local) * 0.5

x_local_std = (x_local - x_local.mean()) / x_local.std()

# Fit polynomial (degree 8) - x_local_std is already numpy
X_poly_local = np.column_stack([x_local_std**p for p in range(1, 9)])
result_poly_local = fit_glm(X=X_poly_local, y=y_local, family='gaussian')

# Fit GAM
result_gam_local = fit_gam(
    x=x_local_std,
    y=y_local,
    n_basis=12,
    basis_type='cubic'
)

# Plot
sort_idx_local = np.argsort(x_local)
plt.figure(figsize=(12, 6))
plt.scatter(x_local, y_local, alpha=0.4, s=30, color='gray', label='Data')
plt.plot(x_local[sort_idx_local], y_true_local[sort_idx_local], 
         'k-', lw=3, label='True (linear + local bump)', zorder=5)
plt.plot(x_local[sort_idx_local], result_poly_local.predict(X_poly_local)[sort_idx_local],
         'r-', lw=2, label='Polynomial (deg 8)', alpha=0.7)
plt.plot(x_local[sort_idx_local], result_gam_local.fitted_values[sort_idx_local],
         'g-', lw=2, label='GAM', alpha=0.7)
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Local Flexibility: GAM Captures Local Features Better\n(Polynomial struggles with local bump without global wiggling)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("GAM handles local features (bump) without global oscillations")
print("Polynomial must wiggle globally to capture local features")

### 2. Automatic Complexity Selection

In [ ]:
print("Model Complexity:")
print("\n📊 Polynomial:")
print("  • Must manually choose degree (2, 3, 4, 5, ...)")
print("  • Try multiple degrees and compare AIC/BIC")
print("  • Risk of overfitting (too high) or underfitting (too low)")
print("  • No built-in regularization")
print("\n📈 GAM:")
print("  • Automatically selects smoothness via GCV")
print("  • Penalized fitting balances fit and smoothness")
print("  • EDF (effective degrees of freedom) measures complexity")
print("  • Built-in regularization prevents overfitting")
print("\n GAM is more automated and robust!")

## 9. When to Use Each Method

### Decision Guide

In [ ]:
decision_guide = pd.DataFrame({
    'Criterion': [
        'Relationship type',
        'Interpretability',
        'Extrapolation',
        'Local features',
        'Model selection',
        'Numerical stability',
        'Computational cost',
        'Multiple predictors'
    ],
    'Polynomial': [
        'Simple global curvature',
        'Moderate (low degree only)',
        'Poor (oscillates)',
        'Requires high degree',
        'Manual (try multiple degrees)',
        'Poor (high degree)',
        'Fast',
        'Interactions complex'
    ],
    'GAM': [
        'Complex local patterns',
        'Good (smooth curves)',
        'Better (local control)',
        'Naturally handles',
        'Automatic (GCV)',
        'Excellent',
        'Moderate',
        'Natural (additive)'
    ]
})

print("\nWhen to Use Each Method:")
print("="*80)
print(decision_guide.to_string(index=False))
print("="*80)

### Practical Recommendations

In [ ]:
print("\n📌 Practical Recommendations:\n")
print("Use POLYNOMIAL when:")
print("  ✓ Relationship is simple (quadratic, cubic)")
print("  ✓ You need exact functional form (e.g., physics model)")
print("  ✓ Interpretability is critical")
print("  ✓ Sample size is very small (< 50)")
print("  ✓ Low degree suffices (2-3)")
print("\nUse GAM when:")
print("  ✓ Relationship is complex or unknown")
print("  ✓ You need local flexibility")
print("  ✓ Multiple non-linear predictors")
print("  ✓ Want automatic smoothness selection")
print("  ✓ Extrapolation is important")
print("  ✓ High-degree polynomial would be needed (≥4)")
print("\n💡 General advice: Start with low-degree polynomial (2-3).")
print("   If inadequate, switch to GAM rather than increasing polynomial degree!")

## Key Takeaways

- **Low-degree polynomials** (2-3) are interpretable and useful
- **High-degree polynomials** (≥5) are usually problematic
- **GAM** provides better balance between flexibility and stability
- **GCV** in GAM automates smoothness selection (no manual tuning!)
- **Local control** is GAM's superpower (polynomials are global)

## Next Steps

- **Classification:** See `02_classification/01_logistic_regression.ipynb`
- **Multiple smooths:** See `02_classification/03_gam_classification.ipynb`
- **Tensor smooths (2D):** See `05_advanced_topics/01_tensor_smooths.ipynb`

## Resources

- [Aurora-GLM Documentation](https://github.com/Matcraft94/Aurora-GLM)
- [Wood (2017) - Generalized Additive Models](https://www.routledge.com/Generalized-Additive-Models-An-Introduction-with-R-Second-Edition/Wood/p/book/9781498728331)
- [Hastie & Tibshirani (1990) - Generalized Additive Models](https://projecteuclid.org/ebooks/monographs-on-statistics-and-applied-probability/Generalized-Additive-Models/toc/10.1214/9781315141824)